# 05 - Visualizacao da Segmentacao Bruta

Le as tabelas analiticas persistidas pelo notebook 04, reconstrói a base linha a linha em memoria e gera um relatorio PDF mais sucinto, focado em estatistica descritiva, associacao, correlacao e regressao simples.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
plt.rcParams['figure.max_open_warning'] = 0
import pandas as pd
from IPython.display import display

root_dir = Path.cwd()
if not (root_dir / 'src').exists() and (root_dir.parent / 'src').exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.analysis import MetricsCollector
from src.io.path_resolver import PathResolver
from src.repositories import (
    AnaliseSegmentacaoBrutaInteracaoTagModeloRepository,
    AnaliseSegmentacaoBrutaIntervaloConfiancaRepository,
    AnaliseSegmentacaoBrutaResumoModeloRepository,
)
from src.visualization import (
    PdfReportSection,
    plot_metric_bars_with_ci_by_model,
    plot_metric_correlation_heatmap,
    plot_metric_distribution_by_model,
    plot_metric_scatter,
    plot_model_tag_interaction_heatmap,
    plot_simple_regression,
    save_pdf_report,
)


## Carregamento das tabelas analiticas e da base linha a linha

A visualizacao agora usa apenas o que e mais util para leitura do projeto: resumos por modelo, intervalos de confianca, interacoes com dificuldade e a base linha a linha para correlacao e regressao.


In [ ]:
def _registros_para_dataframe(registros, columns):
    return pd.DataFrame([{column: getattr(registro, column) for column in columns} for registro in registros], columns=columns)

metric_names = ['auprc', 'soft_dice', 'brier_score']
metric_pairs = [('auprc', 'soft_dice'), ('auprc', 'brier_score'), ('soft_dice', 'brier_score')]
regression_metrics = ['soft_dice', 'brier_score']
interaction_metrics = ['auprc', 'soft_dice', 'brier_score']
path_resolver = PathResolver.from_config()
report_output_path = Path(path_resolver.generated_dir) / '05_visualizacao_segmentacao_bruta.pdf'

collector = MetricsCollector(force_recalculate=False)
df_base = collector.collect_all_metrics()

resumo_modelo_repository = AnaliseSegmentacaoBrutaResumoModeloRepository()
intervalo_confianca_repository = AnaliseSegmentacaoBrutaIntervaloConfiancaRepository()
interacao_tag_modelo_repository = AnaliseSegmentacaoBrutaInteracaoTagModeloRepository()

df_resumo_modelo = _registros_para_dataframe(
    resumo_modelo_repository.list(),
    ['nome_modelo', 'metric_name', 'count', 'mean', 'median', 'std', 'min', 'max', 'q1', 'q3', 'iqr', 'higher_is_better'],
)
df_intervalo_confianca = _registros_para_dataframe(
    intervalo_confianca_repository.list(),
    ['nome_modelo', 'metric_name', 'statistic_name', 'count', 'estimate', 'ci_low', 'ci_high', 'confidence_level', 'n_resamples', 'higher_is_better'],
)
df_interacoes_tag_modelo = _registros_para_dataframe(
    interacao_tag_modelo_repository.list(),
    ['nome_modelo', 'tag_name', 'metric_name', 'count_com_tag', 'count_sem_tag', 'mean_com_tag', 'mean_sem_tag', 'median_com_tag', 'median_sem_tag', 'delta_mean', 'delta_median', 'relative_delta_mean', 'adjusted_delta_mean', 'adjusted_delta_median', 'impact_direction', 'higher_is_better'],
)

print(f'Registros da base linha a linha: {len(df_base)}')
print(f'Resumo por modelo: {len(df_resumo_modelo)}')
print(f'Intervalos de confianca: {len(df_intervalo_confianca)}')
print(f'Interacoes modelo x tag: {len(df_interacoes_tag_modelo)}')
print(f'PDF de saida: {report_output_path}')


## Estatistica descritiva univariada

Este bloco resume cada metrica por modelo usando estimativa central e intervalo de confianca, o que torna a comparacao mais informativa do que olhar apenas medias isoladas.


In [ ]:
texto_univariada = 'Este bloco resume cada metrica por modelo usando estimativa central e intervalo de confianca, o que torna a comparacao mais informativa do que olhar apenas medias isoladas.'
figures_univariada = []

display(df_resumo_modelo.pivot(index='nome_modelo', columns='metric_name', values='mean').sort_index())

for metric_name in metric_names:
    fig, _ = plot_metric_bars_with_ci_by_model(df_resumo_modelo, df_intervalo_confianca, metric_name)
    figures_univariada.append(fig)
    plt.show()


## Distribuicoes por modelo

Os boxplots mostram densidade, centralidade e dispersao diretamente na base linha a linha, complementando os intervalos de confianca com a forma da distribuicao.


In [ ]:
texto_distribuicao = 'Os boxplots mostram densidade, centralidade e dispersao diretamente na base linha a linha, complementando os intervalos de confianca com a forma da distribuicao.'
figures_distribuicao = []

for metric_name in metric_names:
    fig, _ = plot_metric_distribution_by_model(df_base, metric_name)
    figures_distribuicao.append(fig)
    plt.show()


## Analise bivariada e correlacao

Aqui avaliamos se as metricas contam historias complementares ou redundantes, usando graficos bivariados e matrizes de correlacao de Pearson e Spearman.


In [ ]:
texto_correlacao = 'Aqui avaliamos se as metricas contam historias complementares ou redundantes, usando graficos bivariados e matrizes de correlacao de Pearson e Spearman.'
figures_correlacao = []

display(df_base[metric_names].corr(method='pearson'))
display(df_base[metric_names].corr(method='spearman'))

for x_metric, y_metric in metric_pairs:
    fig, _ = plot_metric_scatter(df_base, x_metric, y_metric)
    figures_correlacao.append(fig)
    plt.show()

for method in ['pearson', 'spearman']:
    fig, _ = plot_metric_correlation_heatmap(df_base, metric_names, method)
    figures_correlacao.append(fig)
    plt.show()


## Regressao simples

A regressao simples usa `num_tags_problema` como proxy de dificuldade para medir a tendencia de degradacao das metricas mais interpretaveis para a segmentacao bruta.


In [ ]:
texto_regressao = 'A regressao simples usa num_tags_problema como proxy de dificuldade para medir a tendencia de degradacao das metricas mais interpretaveis para a segmentacao bruta.'
figures_regressao = []

for metric_name in regression_metrics:
    fig, _ = plot_simple_regression(df_base, 'num_tags_problema', metric_name)
    figures_regressao.append(fig)
    plt.show()


## Interacao entre modelo e dificuldade

Para manter o relatorio sucinto, este bloco mostra as tres metricas brutas principais para enxergar queda de qualidade com dificuldade: `auprc`, `soft_dice` e `brier_score`.


In [ ]:
texto_interacoes = 'Para manter o relatorio sucinto, este bloco mostra as tres metricas brutas principais para enxergar queda de qualidade com dificuldade: auprc, soft_dice e brier_score.'
figures_interacoes = []

display(df_interacoes_tag_modelo.sort_values(['metric_name', 'tag_name', 'adjusted_delta_mean']).head(20))

for metric_name in interaction_metrics:
    fig, _ = plot_model_tag_interaction_heatmap(df_interacoes_tag_modelo, metric_name)
    figures_interacoes.append(fig)
    plt.show()


## Leitura inicial

O notebook 05 agora prioriza menos paineis e mais leitura analitica: univariada com intervalo de confianca, distribuicao, associacao entre metricas, correlacao, regressao simples e relacao entre modelo e dificuldade.


In [ ]:
report_sections = [
    PdfReportSection(
        heading='Carregamento das tabelas analiticas e da base linha a linha',
        body='A visualizacao agora usa apenas o que e mais util para leitura do projeto: resumos por modelo, intervalos de confianca, interacoes com dificuldade e a base linha a linha para correlacao e regressao.',
    ),
    PdfReportSection(heading='Estatistica descritiva univariada', body=texto_univariada, figures=figures_univariada),
    PdfReportSection(heading='Distribuicoes por modelo', body=texto_distribuicao, figures=figures_distribuicao),
    PdfReportSection(heading='Analise bivariada e correlacao', body=texto_correlacao, figures=figures_correlacao),
    PdfReportSection(heading='Regressao simples', body=texto_regressao, figures=figures_regressao),
    PdfReportSection(heading='Interacao entre modelo e dificuldade', body=texto_interacoes, figures=figures_interacoes),
    PdfReportSection(
        heading='Leitura inicial',
        body='O notebook 05 agora prioriza menos paineis e mais leitura analitica: univariada com intervalo de confianca, distribuicao, associacao entre metricas, correlacao, regressao simples e relacao entre modelo e dificuldade.',
    ),
]

pdf_path = save_pdf_report(
    output_path=report_output_path,
    sections=report_sections,
    report_title='05 - Visualizacao da Segmentacao Bruta',
)

print(f'Relatorio PDF salvo em: {pdf_path}')
